# Unit 3 Assignment: Production Advanced RAG System

Pipeline implemented:
1. Query Expansion (HyDE with Gemini, temperature 0.0)
2. Hybrid Retrieval (BM25 + SBERT + RRF)
3. Cross-Encoder Re-Ranking
4. Final Answer Generation
5. Naive vs Advanced comparison experiment

## 1) Dependency Installation and Imports
Install required libraries for this assignment.

In [1]:
%pip install -q rank-bm25 sentence-transformers langchain-google-genai langchain-core numpy pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from typing import Any

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

np.random.seed(42)
pd.set_option("display.max_colwidth", 120)
print("Imports loaded.")

c:\Users\Kartik Palcharla\ml_lab\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded.


## 2) Environment Variables and API Client Setup
Load `.env`, initialize Gemini, and handle missing keys explicitly.

In [ ]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = None
if not GOOGLE_API_KEY:
    print("WARNING: GOOGLE_API_KEY not found in .env. HyDE and final LLM generation will use deterministic fallback text.")
else:
    model_candidates = ["gemini-3-flash-preview"]
    for model_name in model_candidates:
        try:
            llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)
            _ = llm.invoke("Reply with one word: ready")
            print(f"Gemini client initialized with model: {model_name}")
            break
        except Exception:
            llm = None
    if llm is None:
        print("WARNING: Gemini init failed for all candidate models. Fallback mode enabled.")

print("GROQ_API_KEY present:", bool(GROQ_API_KEY))

Gemini client initialized with model: gemini-3-flash-preview
GROQ_API_KEY present: True


## 3) Corpus Construction (>=10 AI/ML Documents)
Includes related training sub-topics and jargon-heavy entries for sparse retrieval stress testing.

In [4]:
corpus = [
    "Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence.",
    "Multi-head attention lets a transformer focus on multiple relation patterns at once, such as syntax and long-range dependencies.",
    "Positional encodings inject token order information so attention layers can distinguish different word positions.",
    "Gradient descent updates model weights by moving in the direction that reduces training loss on mini-batches.",
    "Adam combines momentum and adaptive learning rates, often speeding up convergence for deep neural networks.",
    "Learning-rate warmup and cosine decay stabilize early training and improve final generalization in large models.",
    "Regularization methods like dropout and weight decay reduce overfitting by constraining effective model capacity.",
    "Batch normalization and layer normalization reduce internal covariate shift and make optimization more stable.",
    "Backpropagation applies the chain rule to compute gradients efficiently from output layers to earlier layers.",
    "FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes.",
    "Cross-encoder rerankers jointly score query-document pairs and usually improve precision at top ranks.",
    "CUDA kernels and cuDNN accelerate tensor operations on NVIDIA GPUs during neural network training.",
    "Vaswani et al. introduced the Transformer architecture in Attention Is All You Need, replacing recurrence with attention."
]

print(f"Corpus size: {len(corpus)}")
for i, d in enumerate(corpus[:5]):
    print(f"{i}: {d[:90]}...")

Corpus size: 13
0: Transformers represent token meaning using contextual embeddings built from self-attention...
1: Multi-head attention lets a transformer focus on multiple relation patterns at once, such ...
2: Positional encodings inject token order information so attention layers can distinguish di...
3: Gradient descent updates model weights by moving in the direction that reduces training lo...
4: Adam combines momentum and adaptive learning rates, often speeding up convergence for deep...


## 4) Text Normalization and Shared Helper Functions

In [5]:
def tokenize_for_bm25(text: str) -> list[str]:
    return text.lower().split()


def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-12, norms)
    return vectors / norms


def cosine_scores(query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    q = query_vec / (np.linalg.norm(query_vec) + 1e-12)
    return doc_vecs @ q


def rank_map(sorted_doc_ids: np.ndarray) -> dict[int, int]:
    return {int(doc_id): int(rank + 1) for rank, doc_id in enumerate(sorted_doc_ids)}


def rrf_score(doc_id: int, bm25_ranks: dict[int, int], sbert_ranks: dict[int, int], k: int = 60) -> float:
    return (1.0 / (k + bm25_ranks[int(doc_id)])) + (1.0 / (k + sbert_ranks[int(doc_id)]))


def format_doc(doc: dict[str, Any]) -> str:
    return f"[doc_id={doc['doc_id']}] {doc['text']}"

print("Helpers ready.")

Helpers ready.


## 5) `HybridRetriever` Implementation (BM25 + SBERT + RRF)

In [6]:
class HybridRetriever:
    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k
        self.tokenized_corpus = [tokenize_for_bm25(doc) for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        self.bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        raw_doc_vecs = self.bi_encoder.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = l2_normalize(raw_doc_vecs)

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        bm25_scores = self.bm25.get_scores(tokenize_for_bm25(query))
        bm25_sorted = np.argsort(bm25_scores)[::-1]

        q_vec = self.bi_encoder.encode([query], convert_to_numpy=True)[0]
        sbert_scores = cosine_scores(q_vec, self.doc_vecs)
        sbert_sorted = np.argsort(sbert_scores)[::-1]

        bm25_ranks = rank_map(bm25_sorted)
        sbert_ranks = rank_map(sbert_sorted)

        fused = []
        for doc_id in range(len(self.corpus)):
            fused.append({
                "doc_id": int(doc_id),
                "rrf_score": float(rrf_score(doc_id, bm25_ranks, sbert_ranks, self.k)),
                "bm25_rank": int(bm25_ranks[doc_id]),
                "sbert_rank": int(sbert_ranks[doc_id]),
                "text": self.corpus[doc_id],
            })

        fused_sorted = sorted(fused, key=lambda x: x["rrf_score"], reverse=True)
        return fused_sorted[:top_k]

hybrid = HybridRetriever(corpus=corpus, k=60)
print("HybridRetriever initialized.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7975.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HybridRetriever initialized.


## 6) Hybrid Retrieval Smoke Tests and Rank Contribution Checks

In [7]:
diagnostic_queries = [
    "what is attention in transformers",
    "techniques to stabilize neural network training",
    "what does cuDNN do"
]

for q in diagnostic_queries:
    print("\n" + "=" * 80)
    print(f"Query: {q}")
    hits = hybrid.retrieve(q, top_k=5)
    for rank, h in enumerate(hits, start=1):
        print(
            f"{rank}. doc_id={h['doc_id']} | rrf={h['rrf_score']:.6f} | "
            f"bm25_rank={h['bm25_rank']} | sbert_rank={h['sbert_rank']}"
        )
        print(f"   {h['text']}")


Query: what is attention in transformers
1. doc_id=12 | rrf=0.032787 | bm25_rank=1 | sbert_rank=1
   Vaswani et al. introduced the Transformer architecture in Attention Is All You Need, replacing recurrence with attention.
2. doc_id=0 | rrf=0.032002 | bm25_rank=2 | sbert_rank=3
   Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence.
3. doc_id=1 | rrf=0.031754 | bm25_rank=4 | sbert_rank=2
   Multi-head attention lets a transformer focus on multiple relation patterns at once, such as syntax and long-range dependencies.
4. doc_id=2 | rrf=0.031498 | bm25_rank=3 | sbert_rank=4
   Positional encodings inject token order information so attention layers can distinguish different word positions.
5. doc_id=5 | rrf=0.030090 | bm25_rank=5 | sbert_rank=8
   Learning-rate warmup and cosine decay stabilize early training and improve final generalization in large models.

Query: techniques to stabilize neural network training
1. 

## 7) Cross-Encoder `rerank()` Implementation

In [8]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    enriched = []
    for c, s in zip(candidates, scores):
        row = dict(c)
        row["cross_encoder_score"] = float(s)
        enriched.append(row)

    reranked = sorted(enriched, key=lambda x: x["cross_encoder_score"], reverse=True)
    return reranked[:top_k]

# quick rerank check
sample_candidates = hybrid.retrieve("how does transformer attention work", top_k=5)
sample_reranked = rerank("how does transformer attention work", sample_candidates, top_k=3)
print("Reranker loaded. Top 3 reranked docs:")
for r in sample_reranked:
    print(f"doc_id={r['doc_id']} | ce_score={r['cross_encoder_score']:.4f} | {r['text'][:90]}...")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6572.97it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded. Top 3 reranked docs:
doc_id=1 | ce_score=6.6541 | Multi-head attention lets a transformer focus on multiple relation patterns at once, such ...
doc_id=12 | ce_score=3.4952 | Vaswani et al. introduced the Transformer architecture in Attention Is All You Need, repla...
doc_id=2 | ce_score=-8.5871 | Positional encodings inject token order information so attention layers can distinguish di...


## 8) Query Expansion with HyDE (Gemini, `temperature=0.0`)

In [11]:
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert academic assistant. Write one concise factual paragraph that would likely answer the user query."),
    ("human", "Query: {query}")
])
hyde_chain = hyde_prompt | llm | StrOutputParser() if llm is not None else None


def generate_hyde_query(user_query: str) -> str:
    if hyde_chain is None:
        # Deterministic fallback when API is unavailable.
        return f"Technical explanation for: {user_query}. Include precise ML terminology and methods."
    try:
        return hyde_chain.invoke({"query": user_query}).strip()
    except Exception as e:
        print(f"WARNING: HyDE generation failed, fallback used: {e}")
        return f"Technical explanation for: {user_query}. Include precise ML terminology and methods."

print(generate_hyde_query("how do transformers encode meaning?")[:300])

Transformers encode meaning by transforming discrete input tokens into high-dimensional vector embeddings that are iteratively refined through a multi-layered self-attention mechanism. Unlike previous sequential models, transformers use self-attention to compute weighted relationships between every 


## 9) Naive RAG Baseline (Dense-Only SBERT Retrieval)

In [12]:
def naive_rag_top_doc(query: str) -> dict:
    q_vec = hybrid.bi_encoder.encode([query], convert_to_numpy=True)[0]
    scores = cosine_scores(q_vec, hybrid.doc_vecs)
    top_id = int(np.argmax(scores))
    return {
        "doc_id": top_id,
        "score": float(scores[top_id]),
        "text": corpus[top_id],
    }

print(naive_rag_top_doc("optimization techniques for training"))

{'doc_id': 3, 'score': 0.504916250705719, 'text': 'Gradient descent updates model weights by moving in the direction that reduces training loss on mini-batches.'}


## 10) `advanced_rag(user_query)` End-to-End Pipeline

In [13]:
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise university AI assistant. Answer only from context and say when context is insufficient."),
    ("human", "Question: {question}\n\nContext:\n{context}\n\nAnswer in 4-6 sentences.")
])
generation_chain = generation_prompt | llm | StrOutputParser() if llm is not None else None


def advanced_rag_with_debug(user_query: str) -> tuple[str, dict]:
    expanded_query = generate_hyde_query(user_query)

    candidates = hybrid.retrieve(expanded_query, top_k=6)
    reranked = rerank(user_query, candidates, top_k=3)

    context = "\n\n".join([format_doc(d) for d in reranked])
    if generation_chain is None:
        answer = (
            "API key unavailable, so this is a deterministic fallback answer. "
            "Top retrieved evidence suggests: " + reranked[0]["text"]
        )
    else:
        try:
            answer = generation_chain.invoke({"question": user_query, "context": context})
        except Exception as e:
            answer = f"Generation failed ({e}). Best evidence: {reranked[0]['text']}"

    debug = {
        "user_query": user_query,
        "expanded_query": expanded_query,
        "hybrid_candidates": candidates,
        "reranked_top": reranked,
        "used_context": context,
    }
    return answer, debug


def advanced_rag(user_query: str) -> str:
    answer, _ = advanced_rag_with_debug(user_query)
    return answer

ans, dbg = advanced_rag_with_debug("how do transformers encode meaning?")
print("Answer preview:\n", ans[:500])
print("\nTop reranked doc:")
print(dbg["reranked_top"][0]["text"])

Answer preview:
 Transformers represent token meaning through contextual embeddings built from self-attention over all positions in a sequence. Multi-head attention enhances this process by allowing the model to focus on multiple relation patterns simultaneously, such as syntax and long-range dependencies. To distinguish between different word positions, positional encodings are used to inject token order information. This ensures that attention layers can effectively incorporate sequence structure into the mean

Top reranked doc:
Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence.


## 11) Comparison Experiment for 3 Queries (Naive vs Advanced)

In [14]:
test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "why is faiss useful for retrieval systems?"
]

rows = []
for q in test_queries:
    naive = naive_rag_top_doc(q)
    _, debug = advanced_rag_with_debug(q)
    advanced_top = debug["reranked_top"][0]

    rows.append({
        "Query": q,
        "Naive RAG Top Doc": naive["text"],
        "Advanced RAG Top Doc": advanced_top["text"],
        "Are they different?": "Yes" if naive["text"] != advanced_top["text"] else "No",
    })

comparison_df = pd.DataFrame(rows)
comparison_df

,Query,Naive RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a s...,Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a s...,No
1,optimization techniques for training,Gradient descent updates model weights by moving in the direction that reduces training loss on mini-batches.,Batch normalization and layer normalization reduce internal covariate shift and make optimization more stable.,Yes
2,why is faiss useful for retrieval systems?,FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes.,FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes.,No


In [17]:
print(comparison_df.to_dict(orient='records'))

[{'Query': 'how do transformers encode meaning?', 'Naive RAG Top Doc': 'Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence.', 'Advanced RAG Top Doc': 'Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence.', 'Are they different?': 'No'}, {'Query': 'optimization techniques for training', 'Naive RAG Top Doc': 'Gradient descent updates model weights by moving in the direction that reduces training loss on mini-batches.', 'Advanced RAG Top Doc': 'Batch normalization and layer normalization reduce internal covariate shift and make optimization more stable.', 'Are they different?': 'Yes'}, {'Query': 'why is faiss useful for retrieval systems?', 'Naive RAG Top Doc': 'FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes.', 'Advanced RAG Top Doc': 'FAISS enables fast approximate nearest-neighbor vector search for larg

In [15]:
print("Observations:")
for _, row in comparison_df.iterrows():
    marker = "different" if row['Are they different?'] == 'Yes' else 'same'
    print(f"- Query: {row['Query']} -> Top docs are {marker}.")

print("\nExample full advanced answer:")
print(advanced_rag("optimization techniques for training"))

Observations:
- Query: how do transformers encode meaning? -> Top docs are same.
- Query: optimization techniques for training -> Top docs are different.
- Query: why is faiss useful for retrieval systems? -> Top docs are same.

Example full advanced answer:
Optimization techniques for training include gradient descent, which updates model weights by moving in the direction that reduces training loss on mini-batches [doc_id=3]. Batch normalization and layer normalization are also used to reduce internal covariate shift and make optimization more stable [doc_id=7]. Additionally, learning-rate warmup and cosine decay are employed to stabilize early training and improve final generalization in large models [doc_id=5]. These methods collectively address weight updates, internal stability, and learning rate scheduling. The provided context is insufficient to describe other specific optimization algorithms or regularization techniques.


### Filled Comparison Table (Markdown)

| Query | Naive RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| "how do transformers encode meaning?" | Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence. | Transformers represent token meaning using contextual embeddings built from self-attention over all positions in a sequence. | No |
| "optimization techniques for training" | Gradient descent updates model weights by moving in the direction that reduces training loss on mini-batches. | Batch normalization and layer normalization reduce internal covariate shift and make optimization more stable. | Yes |
| "why is faiss useful for retrieval systems?" | FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes. | FAISS enables fast approximate nearest-neighbor vector search for large embedding indexes. | No |